### 🎯 Module Overview
This module covers everything you need to know about parsing and ingesting data for RAG systems, from basic text files to complex PDFs and databases. We'll use LangChain v0.3 and explore each technique with practical examples.

Table of Contents
- Introduction to Data Ingestion
- Text Files (.txt)
- PDF Documents
- Microsoft Word Documents
- CSV and Excel Files
- JSON and Structured Data
- Web Scraping
- Databases (SQL)
- Audio and Video Transcripts
- Advanced Techniques
- Best Practices

### This Notebook overview:

- Introduction to Data Ingestion
- Understanding Document Structure In Langchain
- Text Files (.txt)
- TextLoader- Read single fiole
- DirectoryLoader- Read multiple files in a directory
- Text splitting strategies

### Introduction To Data Ingestion

In [4]:
import os
from typing import List, Dict, Any
import pandas as pd

In [5]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
)

print("Set up Completed!")

d:\VM_Shared\codes\myCodes\ultimateRAGBootcamp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Set up Completed!


### Understanding Document Structure In Langchain

In [6]:
## create a simple document
doc = Document(
    page_content="This is the main text content that will be embedded and searched.",
    metadata={
        "source": "example.txt",
        "page": 1,
        "author": "Krish Naik",
        "date_created": "2024-01-01",
        "custom_field": "any_value",
    },
)
print("Document Structure")

print(f"Content :{doc.page_content}")
print(f"Metadata :{doc.metadata}")

# Why metadata matters:
print("\n📝 Metadata is crucial for:")
print("- Filtering search results")
print("- Tracking document sources")
print("- Providing context in responses")
print("- Debugging and auditing")

Document Structure
Content :This is the main text content that will be embedded and searched.
Metadata :{'source': 'example.txt', 'page': 1, 'author': 'Krish Naik', 'date_created': '2024-01-01', 'custom_field': 'any_value'}

📝 Metadata is crucial for:
- Filtering search results
- Tracking document sources
- Providing context in responses
- Debugging and auditing


In [7]:
type(doc)

langchain_core.documents.base.Document

### Text Files (.txt) - The Simplest Case {#2-text-files}

In [8]:
## Create a simple txt file
import os

os.makedirs("data/text_files", exist_ok=True)

In [9]:
sample_texts = {
    "data/text_files/python_intro.txt": """Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    "data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """,
}

for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


### Document Loaders

#### TextLoader- Read Single File 

In [10]:
# from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader

## Loading a single text file
loader = TextLoader("data/text_files/python_intro.txt", encoding="utf-8")

documents = loader.load()
print(f"📄 Loaded {len(documents)} document")
print(f"Content preview: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")


📄 Loaded 1 document
Content preview: Python Programming Introduction

Python is a high-level, interpreted programming language known for ...
Metadata: {'source': 'data/text_files/python_intro.txt'}


#### DirectoryLoader- Multiple Text Files

In [11]:
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt",  # Pattern to match files
    loader_cls=TextLoader,  # loader class to use
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
)

documents = dir_loader.load()

print(f"📁 Loaded {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"\nDocument {i + 1}:")
    print(f"  Source: {doc.metadata['source']}")
    print(f"  Length: {len(doc.page_content)} characters")


# 📊 Analysis
print("\n📊 DirectoryLoader Characteristics:")
print("✅ Advantages:")
print("  - Loads multiple files at once")
print("  - Supports glob patterns")
print("  - Progress tracking")
print("  - Recursive directory scanning")

print("\n❌ Disadvantages:")
print("  - All files must be same type")
print("  - Limited error handling per file")
print("  - Can be memory intensive for large directories")

100%|██████████| 2/2 [00:00<00:00, 999.95it/s]

📁 Loaded 2 documents

Document 1:
  Source: data\text_files\machine_learning.txt
  Length: 575 characters

Document 2:
  Source: data\text_files\python_intro.txt
  Length: 489 characters

📊 DirectoryLoader Characteristics:
✅ Advantages:
  - Loads multiple files at once
  - Supports glob patterns
  - Progress tracking
  - Recursive directory scanning

❌ Disadvantages:
  - All files must be same type
  - Limited error handling per file
  - Can be memory intensive for large directories


### Text Splitting

- Splitting text into manageable chunks is crucial for effective processing in RAG systems.
- This ensures that the language model can handle the input without exceeding token limits and allows for more focused retrieval of relevant information.
- Choosing the right splitting strategy depends on the structure of your data and the specific use case.

#### Splitting Statergies


In LangChain following text splitters are available:

- `CharacterTextSplitter`: Splits text based on character count, with options to split at newlines or spaces.
- `RecursiveCharacterTextSplitter`: Splits text recursively, prioritizing larger chunks (like paragraphs) before smaller ones (like lines or spaces).
- `TokenTextSplitter`: Splits text based on token count, which is more aligned with how language models process input.


#### Comparison: Newline vs. Whitespace separators

| Feature | Separator: "\n" (Newline) | Separator: " " (Whitespace) |
|---|---|---|
| Granularity | Splits at line breaks or paragraph ends. | Splits between individual words. |
| Context | Better for keeping related lines or bullet points together. | High risk of cutting sentences or phrases mid-thought. |
| Chunk Size Control | May result in chunks larger than your `chunk_size` if a single line is too long. | Provides more precise control over `chunk_size` as words are small units. |
| Best For | Structured text like logs, lists, or distinct paragraphs. | Very short, unstructured text where keeping sentences whole isn't a priority. |
| Behavior | It waits for a newline. Even though chunk_size is 20, it won't break the first line (43 chars) because there is no \n until the end. | It splits at the first space it finds after reaching the chunk_size. This ignores the structure of the newlines entirely. |
| Output simplified | `['This is a long sentence that stays together.', 'This is a new line.', 'And another one.']` | `['This is a long', 'sentence that stays', 'together.\nThis is a', ...]` |

#### Key Differences in Behavior

* Default Behavior: By default, CharacterTextSplitter uses "\n\n" (double newline) to target paragraphs.
* The "Strict" Rule: This splitter only splits on the specific character you provide. If you use "\n" and your text has a 1000-character line, but your `chunk_size` is 500, it will still output a 1000-character chunk because it cannot find a newline to break it.
* Whitespace Fragmentation: Using " " as a separator effectively treats your text as a bag of words. While this ensures your chunks stay close to the `chunk_size` limit, it often leads to "fragmented" chunks that are hard for an LLM to understand because context is lost mid-sentence.

#### Pro-Tip: Use RecursiveCharacterTextSplitter Instead 

Unless you have a very specific reason to use CharacterTextSplitter, most developers prefer the [RecursiveCharacterTextSplitter](https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter).
It uses a prioritized list of separators: `["\n\n", "\n", " ", ""]`. It tries to split by paragraphs first; if that's too big, it tries newlines; if still too big, it tries spaces. This approach keeps paragraphs and sentences together as much as possible while strictly honoring your `chunk_size`.

In [12]:
# Different text splitting strategies
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)

print(documents)

[Document(metadata={'source': 'data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '), Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprog

In [13]:
# MEthod 1- Character Text Splitter
text = documents[0].page_content
text

'Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '

#### `CharacterTextSplitter`

In [19]:
# Method 1: Character-based splitting
print("1️⃣ CHARACTER TEXT SPLITTER")
char_splitter = CharacterTextSplitter(
    separator=" ",  # Split on whitespace, newline is preserved in the chunks
    chunk_size=200,  # Max chunk size in characters
    chunk_overlap=20,  # Overlap between chunks
    length_function=len,  # How to measure chunk size
)

char_chunks = char_splitter.split_text(text)
print(type(char_chunks), type(char_chunks[0]))
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")
print("all chunks:")
print("\n------\n".join(char_chunks))

1️⃣ CHARACTER TEXT SPLITTER
<class 'list'> <class 'str'>
Created 3 chunks
First chunk: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...
all chunks:
Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing
------
on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning:
------
Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems


In [20]:
# Method 1: Character-based splitting
print("1️⃣ CHARACTER TEXT SPLITTER")
char_splitter = CharacterTextSplitter(
    separator="\n",  # Split on newlines
    chunk_size=200,  # Max chunk size in characters
    chunk_overlap=20,  # Overlap between chunks
    length_function=len,  # How to measure chunk size
)

char_chunks = char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk:\n{char_chunks[0][:100]}...\n")
print("\n-------\n".join(char_chunks))


1️⃣ CHARACTER TEXT SPLITTER
Created 4 chunks
First chunk:
Machine Learning Basics
Machine learning is a subset of artificial intelligence that enables systems...

Machine Learning Basics
Machine learning is a subset of artificial intelligence that enables systems to learn and improve
-------
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.
Types of Machine Learning:
-------
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties
-------
Applications include image recognition, speech processing, and recommendation systems


#### `RecursiveCharacterTextSplitter`

In [21]:
# Method 2: Recursive character splitting (RECOMMENDED)
print("\n2️⃣ RECURSIVE CHARACTER TEXT SPLITTER")
default_rec_splitter = RecursiveCharacterTextSplitter(
    # default, breaks paragraphs, then lines, then spaces, then characters
    separators=["\n\n", "\n", " ", ""],
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
)
paragraph_rec_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n"],  # breaks on paragraph
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
)
new_line_rec_splitter = RecursiveCharacterTextSplitter(
    separators=["\n"],  # breaks on line
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
)
ws_rec_splitter = RecursiveCharacterTextSplitter(
    separators=[" "],  # breaks on space
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
)

default_rec_chunks = default_rec_splitter.split_text(text)
paragraph_rec_chunks = paragraph_rec_splitter.split_text(text)
new_line_rec_chunks = new_line_rec_splitter.split_text(text)
ws_rec_chunks = ws_rec_splitter.split_text(text)

print("lengths:")
print(f"default_rec_chunks {len(default_rec_chunks)}")
print(f"paragraph_rec_chunks {len(paragraph_rec_chunks)}")
print(f"new_line_rec_chunks {len(new_line_rec_chunks)}")
print(f"ws_rec_chunks {len(ws_rec_chunks)}")


2️⃣ RECURSIVE CHARACTER TEXT SPLITTER
lengths:
default_rec_chunks 6
paragraph_rec_chunks 4
new_line_rec_chunks 4
ws_rec_chunks 4


##### Default chunking with RecursiveCharacterTextSplitter

(paragraphs, then lines, then spaces):

- It splits the text into chunks, keeping paragraphs intact when possible. 
- The first chunk is a full paragraph, the second chunk is another full paragraph, and the third and fourth chunks are split at newlines and spaces as needed to respect the `chunk_size` limit.

In [22]:
print(f"default_rec_chunks {'\n-------\n'.join(default_rec_chunks)}")

default_rec_chunks Machine Learning Basics
-------
Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
-------
that can access data and use it to learn for themselves.
-------
Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
-------
3. Reinforcement Learning: Learning through rewards and penalties
-------
Applications include image recognition, speech processing, and recommendation systems


##### paragraph chunking with RecursiveCharacterTextSplitter:
- It prioritizes splitting at paragraphs. The first two chunks are full paragraphs, and the third chunk is split at a newline because the paragraph exceeds the `chunk_size`.

In [23]:
print(f"paragraph_rec_chunks {'\n-------\n'.join(paragraph_rec_chunks)}")

paragraph_rec_chunks Machine Learning Basics
-------


Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.
-------


Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties
-------
Applications include image recognition, speech processing, and recommendation systems


##### new line chunking with RecursiveCharacterTextSplitter:
- It prioritizes splitting at newlines. The first chunk is a full paragraph
- the second chunk is split at a newline, and the third chunk is split at a space to respect the `chunk_size`.

In [24]:
print(f"new_line_rec_chunks {'\n-------\n'.join(new_line_rec_chunks)}")

new_line_rec_chunks Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
-------
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
-------
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties
-------
Applications include image recognition, speech processing, and recommendation systems


##### whitespace chunking with RecursiveCharacterTextSplitter:
- It prioritizes splitting at spaces. The first chunk is a full paragraph, and the second and third chunks are split at spaces to respect the `chunk_size`, resulting in more fragmented chunks.

In [25]:
print(f"ws_rec_chunks {'\n-------\n'.join(ws_rec_chunks)}")

ws_rec_chunks Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing
-------
on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning:
-------
Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation
-------
and recommendation systems


In [26]:
# Create text without natural break points
simple_text = "This is sentence one and it is quite long. This is sentence two and it is also quite long. This is sentence three which is even longer than the others. This is sentence four. This is sentence five. This is sentence six."

splitter = RecursiveCharacterTextSplitter(
    separators=[" "],  # Only split on spaces
    chunk_size=80,
    chunk_overlap=20,
    length_function=len,
)

chunks = splitter.split_text(simple_text)

print(f"\nSimple text example - {len(chunks)} chunks:\n")

for i in range(len(chunks) - 1):
    print(f"Chunk {i + 1}: '{chunks[i]}'")
    print(f"Chunk {i + 2}: '{chunks[i + 1]}'")

    print()


Simple text example - 4 chunks:

Chunk 1: 'This is sentence one and it is quite long. This is sentence two and it is also'
Chunk 2: 'two and it is also quite long. This is sentence three which is even longer than'

Chunk 2: 'two and it is also quite long. This is sentence three which is even longer than'
Chunk 3: 'is even longer than the others. This is sentence four. This is sentence five.'

Chunk 3: 'is even longer than the others. This is sentence four. This is sentence five.'
Chunk 4: 'is sentence five. This is sentence six.'



#### `TokenTextSplitter`

- `TokenTextSplitter` splits text based on token count rather than character count.
- This is more aligned with how language models process input, as they operate on tokens rather than raw characters.
- It uses a tokenizer (like the one from OpenAI's tiktoken library) to count tokens and split the text accordingly.
- This method is useful when you want to ensure that your chunks are optimized for the token limits of the language model you're using, rather than just character limits.

In [27]:
# Method 3: Token-based splitting
print("\n3️⃣ TOKEN TEXT SPLITTER")
token_splitter = TokenTextSplitter(
    chunk_size=50,  # Size in tokens (not characters)
    chunk_overlap=10,
)

token_chunks = token_splitter.split_text(text)
print(f"Created {len(token_chunks)} chunks")
print(f"First chunk: {token_chunks[0][:100]}...")


3️⃣ TOKEN TEXT SPLITTER


Created 3 chunks
First chunk: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...


#### 📊 Text Splitting Methods Comparison:

##### CharacterTextSplitter:

- ✔️ Simple and predictable
- ✔️ Good for structured text
- ❌ May break mid-sentence
- 🤔 When to use: Text has clear delimiters.

##### RecursiveCharacterTextSplitter:

- ✔️ Respects text structures like paragraphs and sentences to keep related data together
- ✔️ Tries multiple separators
- ✔️ Best general-purpose splitter
- ❌ Slightly more complex
- 🤔 When to use: Default choice for most texts

##### TokenTextSplitter:

- ✔️ Respects model token limits
- ✔️ More accurate for embeddings
- ✔️ Counts every single character as token. (including spaces and punctuation)
- ❌ Slower than character-based
- 🤔 When to use: Working with token-limited models